<a href="https://colab.research.google.com/github/alfiyatunnajah9/Tugas-Machine-Learning/blob/main/Praktikum_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Tugas Mechine Learning 4

4.1 Persiapan Library

In [ ]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

4.2 Import Dataset

In [ ]:
rng = np.random.default_rng(42)
n = 500

df = pd.DataFrame({
    "luas_bangunan": rng.normal(120, 40, n).clip(30, None),
    "jumlah_kamar": rng.integers(1, 6, n),
    "tipe_properti": rng.choice(["Rumah", "Apartemen", "Ruko"], n),
    "kondisi_bangunan": rng.choice(["Buruk", "Sedang", "Baik", "Sangat Baik"], n)
})

df["harga"] = (
    df["luas_bangunan"] * 15 +
    df["jumlah_kamar"] * 20 +
    rng.normal(0, 50, n)
) * 1_000_000

df.head()

4.3 Feature Construction & Preprocessing (ColumnTransformer)

In [ ]:
df["rasio_kamar_per_luas"] = df["jumlah_kamar"] / df["luas_bangunan"]

fitur_numerik = ["luas_bangunan", "jumlah_kamar", "rasio_kamar_per_luas"]
fitur_kategorikal = ["tipe_properti"]
fitur_ordinal = ["kondisi_bangunan"]
urutan_kondisi = [["Buruk", "Sedang", "Baik", "Sangat Baik"]]

preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), fitur_numerik),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), fitur_kategorikal),
    ("ord", OrdinalEncoder(categories=urutan_kondisi), fitur_ordinal)
])

4.4 Training Model (Pipeline Lengkap)

In [ ]:
X = df.drop(columns=["harga"])
y = df["harga"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestRegressor(n_estimators=200, random_state=42))
])

pipeline.fit(X_train, y_train)

4.5 Evaluasi

In [ ]:
y_pred = pipeline.predict(X_test)
print("R2 Score:", round(r2_score(y_test, y_pred), 3))

4.6 Visualisasi

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(y_test, y_pred, alpha=0.6)
plt.xlabel("Harga Aktual")
plt.ylabel("Harga Prediksi")
plt.title("Prediksi vs Aktual (Pipeline Lengkap)")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.show()

4.3 Praktikum 1

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif, RFE, SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

data = load_breast_cancer()
X, y = data.data, data.target
fitur = data.feature_names

X_scaled = StandardScaler().fit_transform(X)

filter_model = SelectKBest(score_func=f_classif, k=5)
filter_model.fit(X, y)
fitur_filter = fitur[filter_model.get_support()]

rfe = RFE(
    estimator=LogisticRegression(max_iter=5000),
    n_features_to_select=5
)
rfe.fit(X_scaled, y)
fitur_rfe = fitur[rfe.get_support()]

lasso = LogisticRegression(
    penalty="l1", solver="liblinear", C=0.1, max_iter=5000
)
embedded = SelectFromModel(lasso)
embedded.fit(X_scaled, y)
fitur_lasso = fitur[embedded.get_support()]

print("Fitur Filter:", list(fitur_filter))
print("Fitur RFE   :", list(fitur_rfe))
print("Fitur Lasso :", list(fitur_lasso))

4.2 Praktikumm 2

In [ ]:
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

df = sns.load_dataset("titanic")

kolom_numerik = ["age", "fare", "pclass"]
kolom_kategorikal = ["sex", "embarked"]

X = df[kolom_numerik + kolom_kategorikal]
y = df["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numerik = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

kategorikal = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numerik, kolom_numerik),
    ("cat", kategorikal, kolom_kategorikal)
])

model = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
akurasi = accuracy_score(y_test, y_pred)
print("Akurasi model (data uji):", round(akurasi, 3))

skor = cross_val_score(model, X, y, cv=5)
print("Akurasi CV (rata-rata)  :", round(skor.mean(), 3))

4.3 Praktiikum

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.feature_selection import SelectFromModel

data = load_breast_cancer()
X, y = data.data, data.target
fitur = data.feature_names

filter_model = SelectKBest(score_func=f_classif, k=5)
filter_model.fit(X, y)
fitur_filter = fitur[filter_model.get_support()]

model_rfe = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=5000)
)
rfe = RFE(
    estimator=LogisticRegression(max_iter=5000),
    n_features_to_select=5
)
rfe.fit(StandardScaler().fit_transform(X), y)
fitur_rfe = fitur[rfe.get_support()]

model_lasso = LogisticRegression(
    penalty="l1",
    solver="liblinear",
    max_iter=5000
)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
model_lasso.fit(X_scaled, y)

lasso = SelectFromModel(
    model_lasso,
    prefit=True,
    threshold="mean"
)
fitur_lasso = fitur[lasso.get_support()]

print("Fitur Filter:", list(fitur_filter))
print("Fitur RFE:", list(fitur_rfe))
print("Fitur Lasso:", list(fitur_lasso))

print("\nJumlah fitur terpilih:")
print("Filter:", len(fitur_filter))
print("RFE:", len(fitur_rfe))
print("Lasso:", len(fitur_lasso))